# Spectrum workflow: ingest, query, ML, and FITS export

This notebook covers the full 1-D spectrum layer:
1. Ingest SDSS/BOSS or DESI spectra into HEALPix-partitioned Zarr v3 stores
2. Update catalog Parquet tiles with `_spectrum_index`
3. Query and visualise spectra via `SpectrumAccessor`
4. Apply built-in transforms (mask, normalise, rest-frame resample)
5. Batch retrieval; extract a curated subset from a FITS target list (`TARGETID`, etc.)
6. Catalog SQL queries, PyTorch training, and FITS export with round-trip verification

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

from data_lake.ingest.fits_to_spectra_zarr import ingest_spectra_from_fits
from data_lake.ingest.update_catalog_indices import update_index_column
from data_lake.io.spectra import SpectrumAccessor
from data_lake.io.catalog import CatalogAccessor
from data_lake.ml.spectrum_dataset import (
    SpectrumDataset, TileSpectrumDataset,
    mask_to_nan, median_normalise, log_flux, rest_frame_resample, compose,
)
from data_lake.export.to_spectrum_fits import export_spectrum, verify_spectrum_round_trip

LAKE_ROOT = Path("/data/lake")   # adjust to your lake root
SURVEY = "sdss_dr17"

## 1. Ingest spectra

The ingestor auto-detects SDSS/BOSS `spec-*.fits`, DESI `coadd-*.fits`, and generic WCS FITS.

It returns a `{source_id: spectrum_index}` map that is then written back to the catalog.

In [ ]:
index_map = ingest_spectra_from_fits(
    source_path="/data/raw/spec-3586-55181-0001.fits",   # SDSS example
    output_root=LAKE_ROOT,
    survey_name=SURVEY,
    wavelength_mode="shared",   # shared wavelength grid (SDSS log-lambda)
)
print(f"Ingested {len(index_map)} spectra")

In [ ]:
# Write _spectrum_index back to the Parquet catalog tiles.
# Note: when using the dl-ingest-spectra CLI this step runs automatically
# (--no-update-catalog to skip).  The ID column is resolved from
# catalog_info.json, so DESI catalogs with TARGETID are handled correctly.
# To backfill an existing lake: dl-rebuild-catalog-indices --survey SURVEY --kind spectrum
n_updated = update_index_column(
    lake_root=LAKE_ROOT,
    survey_name=SURVEY,
    source_id_to_index=index_map,
    kind="spectrum",
)
print(f"Updated {n_updated} Parquet tile file(s)")

## 2. Retrieve a single spectrum

In [ ]:
acc = SpectrumAccessor(LAKE_ROOT, SURVEY)
print(acc)

EXAMPLE_ID = next(iter(index_map))   # use first ingested source
sp = acc.get_spectrum(EXAMPLE_ID)

print(f"source_id : {sp.source_id}")
print(f"n_pix     : {len(sp.flux)}")
print(f"redshift  : z = {sp.meta['z']:.4f}")
print(f"WCS       : {sp.wcs_attrs}")

## 3. Visualise the spectrum

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

# Observed frame
axes[0].plot(sp.wavelength, sp.flux, lw=0.6, color="steelblue", label="flux")
axes[0].fill_between(sp.wavelength,
                     sp.flux - sp.err, sp.flux + sp.err,
                     alpha=0.3, color="steelblue")
# Shade masked pixels
bad = sp.mask != 0
if bad.any():
    axes[0].fill_between(sp.wavelength, *axes[0].get_ylim(),
                          where=bad, color="red", alpha=0.15, label="masked")
axes[0].set_ylabel(r"Flux [10$^{-17}$ erg/s/cm²/Å]")
axes[0].legend()
axes[0].set_title(f"source_id={sp.source_id}  z={sp.meta['z']:.4f}")

# Rest-frame view
axes[1].plot(sp.rest_frame_wavelength(), sp.flux, lw=0.6, color="darkorange")
axes[1].set_xlabel("Wavelength [Å]")
axes[1].set_ylabel(r"Flux (rest frame)")

plt.tight_layout()
plt.show()

## 4. Apply transforms

In [ ]:
# Chain transforms: mask bad pixels → median normalise → log stretch
pipeline = compose(mask_to_nan, median_normalise, log_flux)
sp_processed = pipeline(sp)

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(sp_processed.wavelength, sp_processed.flux, lw=0.6)
ax.set_xlabel("Wavelength [Å]")
ax.set_ylabel("log(1 + norm flux)")
ax.set_title("After mask_to_nan → median_normalise → log_flux")
plt.tight_layout()
plt.show()

In [ ]:
# Rest-frame resample to a common log-lambda grid (e.g. for CNN training)
TARGET_WAVE = np.logspace(np.log10(3700), np.log10(9000), 4000)  # 4000 rest-frame pixels

resample_transform = compose(
    mask_to_nan,
    rest_frame_resample(TARGET_WAVE),
    median_normalise,
)
sp_rf = resample_transform(sp)
print(f"Resampled spectrum shape: {sp_rf.flux.shape}")

## 5. Batch retrieval

In [ ]:
batch_ids = list(index_map.keys())[:10]
flux_batch, ivar_batch, mask_batch, wavelength = acc.get_batch(batch_ids)
print(f"Batch flux shape  : {flux_batch.shape}")
print(f"Wavelength shape  : {wavelength.shape if hasattr(wavelength, 'shape') else 'per-source list'}")

## 6. Extract a curated subset from a FITS target list

Once spectra are ingested into the lake, you can materialise a self-contained Zarr
group containing only sources listed in an external catalog (e.g. a DESI
`zall-pix-*.fits` table with a `TARGETID` column). Reads are batched per HEALPix
tile so each Zarr shard is decompressed at most once.

Lookup uses the Parquet catalog `_spectrum_index` when available (fast); otherwise
it falls back to a vectorised per-tile `source_id` scan.

**Output layout** (`wavelength_mode="shared"` only):

```
qso_subset.zarr/
  flux/        (N_written, N_pix) float32 sharded
  ivar/        (N_written, N_pix) float32 sharded
  mask/        (N_written, N_pix) uint8   sharded
  wavelength/  (N_pix,)           float64 shared grid
  source_id/   (N_written,)       int64
  redshift/    (N_written,)       float32  (per-source z from meta)
```

Equivalent CLI:

```bash
dl-extract-spectra-subset \
    --survey desi_dr1 \
    --target-list zall-pix-iron-qso.fits \
    --target-id-col TARGETID \
    --output /scratch/qso_subset.zarr
```

In [ ]:
from astropy.table import Table

# Adjust paths: any FITS/CSV/Parquet/plain-text list supported by dl-extract-spectra-subset
TARGET_LIST = Path("/data/catalogs/zall-pix-iron-qso.fits")
TARGET_ID_COL = "TARGETID"
SUBSET_ZARR = Path("/scratch/qso_subset.zarr")

target_ids = np.asarray(Table.read(TARGET_LIST)[TARGET_ID_COL])

acc = SpectrumAccessor(LAKE_ROOT, SURVEY)
result = acc.extract_subset_to_zarr(
    source_ids=target_ids,
    output_zarr=SUBSET_ZARR,
    missing="skip",       # report-and-skip IDs not in the lake
    overwrite=False,
)
print(result["n_written"], "rows written;", len(result["missing_ids"]), "missing")
# result["id_to_row"] maps source_id → row index in the flat Zarr if you need reordering

## 7. DuckDB catalog query → spectrum retrieval

In [ ]:
cat = CatalogAccessor(LAKE_ROOT, SURVEY)

# Select low-redshift galaxies with spectra already ingested
df = cat.query("""
    SELECT source_id, ra, dec, _spectrum_index
    FROM catalog
    WHERE _spectrum_index >= 0
      AND z BETWEEN 0.05 AND 0.15
    LIMIT 100
""")
print(f"Found {len(df)} sources with spectra at 0.05 < z < 0.15")
df.head()

## 8. PyTorch training loop

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

TARGET_WAVE_ML = np.logspace(np.log10(3700), np.log10(9000), 4096)

def ml_transform(sp):
    """Mask → rest-frame resample → median normalise → tensor."""
    sp = mask_to_nan(sp)
    sp = rest_frame_resample(TARGET_WAVE_ML)(sp)
    sp = median_normalise(sp)
    flux = np.nan_to_num(sp.flux, nan=0.0)
    return torch.from_numpy(flux.astype(np.float32))

source_ids_train = df["source_id"].tolist()

ds = SpectrumDataset(
    lake_root=LAKE_ROOT,
    survey=SURVEY,
    source_ids=source_ids_train,
    transform=ml_transform,
    catalog_accessor=cat,
)
loader = DataLoader(ds, batch_size=32, shuffle=True, num_workers=2)
print(f"Dataset size: {len(ds)}, batches: {len(loader)}")

In [ ]:
# Minimal 1-D CNN redshift regressor (toy model)
n_pix = len(TARGET_WAVE_ML)
model = nn.Sequential(
    nn.Conv1d(1, 32, kernel_size=7, padding=3), nn.ReLU(),
    nn.MaxPool1d(4),
    nn.Conv1d(32, 64, kernel_size=5, padding=2), nn.ReLU(),
    nn.AdaptiveAvgPool1d(32),
    nn.Flatten(),
    nn.Linear(64 * 32, 64), nn.ReLU(),
    nn.Linear(64, 1),
)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
optimiser = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

for step, batch in enumerate(loader):
    spectra = batch.unsqueeze(1).to(device)     # (B, 1, N_pix)
    dummy_z = torch.zeros(len(batch), 1, device=device)   # replace with real labels
    optimiser.zero_grad()
    pred = model(spectra)
    loss = loss_fn(pred, dummy_z)
    loss.backward()
    optimiser.step()
    if step >= 4:
        break

print(f"Training step loss: {loss.item():.4f}")

## 9. Tile-iterable dataset (full-epoch streaming)

In [ ]:
tile_ds = TileSpectrumDataset(
    lake_root=LAKE_ROOT,
    survey=SURVEY,
    transform=ml_transform,
    shuffle_tiles=True,
)
tile_loader = DataLoader(tile_ds, batch_size=32, num_workers=2)

for step, batch in enumerate(tile_loader):
    print(f"Tile streaming step {step}: batch shape {batch.shape}")
    if step >= 3:
        break

## 10. Export to FITS and verify round-trip

In [ ]:
fits_path = export_spectrum(
    lake_root=LAKE_ROOT,
    survey=SURVEY,
    source_id=EXAMPLE_ID,
    output_path=f"/tmp/spec_{EXAMPLE_ID}.fits",
)
print(f"Written: {fits_path}")

In [ ]:
ok = verify_spectrum_round_trip(LAKE_ROOT, SURVEY, EXAMPLE_ID)
print("Round-trip OK" if ok else "Round-trip FAILED")

In [ ]:
# Inspect the exported FITS structure
from astropy.io import fits as afits
with afits.open(str(fits_path)) as hdul:
    hdul.info()
    print("\nPrimary header (spectral WCS):")
    for k in ("CTYPE1", "CRVAL1", "CDELT1", "CRPIX1", "CUNIT1", "Z", "SOURCE_ID"):
        print(f"  {k:12s} = {hdul[0].header.get(k, 'N/A')}")